# Experiments 

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import mlflow
import shap


from sklearn.preprocessing import PowerTransformer, OneHotEncoder, MultiLabelBinarizer, OrdinalEncoder, MinMaxScaler, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, SimpleImputer, IterativeImputer, MissingIndicator
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_validate
from sklearn.neighbors import LocalOutlierFactor

In [2]:
import dagshub
dagshub.init(repo_owner='bowlekarbhushan88', repo_name='property-price-prediction', mlflow=True)

Accessing as bowlekarbhushan88

Initialized MLflow to track repo "bowlekarbhushan88/property-price-prediction"

Repository bowlekarbhushan88/property-price-prediction initialized!

In [3]:
# set the tracking server

mlflow.set_tracking_uri("https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow")

In [4]:
#Here, I used the data saved using VS Code. This data has 184 columns, which means it is the result of merging df and am_df.
df = pd.read_csv(r'C:\Users\AMD\Desktop\bhushan PC data 11-8-2025/bhushan/property_project/files_vscode/data/py_cleaned_data.csv')

C:\Users\AMD\AppData\Local\Temp\ipykernel_110208\1989270230.py:2: DtypeWarning: Columns (106,128,134,135,136,137,138,139) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\AMD\Desktop\bhushan PC data 11-8-2025/bhushan/property_project/files_vscode/data/py_cleaned_data.csv')


In [5]:
# drop columns not required for model input 
columns_to_drop = ['id','price_category','costpersqft','emi']

df.drop(columns = columns_to_drop , inplace = True)

In [6]:
# combine all amenities and make one column
# Step 1: Filter columns that start with 'am_'
am_cols = [col for col in df.columns if col.startswith('am_')]

# Step 2: Combine values row-wise into a comma-separated string (not a list)
df['amenities'] = df[am_cols].apply(
    lambda row: ', '.join([str(val).strip() for val in row if isinstance(val, str) and val.strip() != ""]),
    axis=1
)

In [7]:
# Step 3: Drop original 'am_' columns
df.drop(columns=am_cols, inplace=True)

In [8]:
#drop duplicate rows 
df.drop_duplicates(inplace=True)

## Data preparation 

In [9]:
temp_df = df.copy()

In [10]:
# split data 

X = temp_df.drop(columns = 'price')
y = temp_df['price']

In [11]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [12]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (9302, 47)
The shape of test data is (2326, 47)


`lets categorized the columns`

## Transform Target column

In [13]:
pt = PowerTransformer(method='yeo-johnson')
y_train_trans = pd.Series(
    pt.fit_transform(y_train.values.reshape(-1, 1)).ravel(),
    index=y_train.index
)
y_test_trans = pd.Series(
    pt.transform(y_test.values.reshape(-1, 1)).ravel(),
    index=y_test.index
)

In [14]:
#o/p in daraframe
from sklearn import set_config
set_config(transform_output="pandas")

`Till this the code is common for all below experimants`

# experiment :04 (Baseline_model + simple LOF + RFECV)

In [15]:
# mlflow experiment

mlflow.set_experiment("Exp 4 - Baseline model,LOF and RFECV")

<Experiment: artifact_location='mlflow-artifacts:/0a91b15efea044f8a8faf281570b81a6', creation_time=1754998139366, experiment_id='5', last_update_time=1754998139366, lifecycle_stage='active', name='Exp 5 - Baseline model,LOF and RFECV', tags={}>

In [16]:
# --- Custom MultiLabel Binarizer ---
class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}
        self.columns = []

    def fit(self, X, y=None):
        self.columns = X.columns
        for col in self.columns:
            mlb = MultiLabelBinarizer()
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            mlb.fit(split_data)
            self.encoders[col] = mlb
        return self

    def transform(self, X):
        output_parts = []
        for col in self.columns:
            mlb = self.encoders[col]
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            transformed = mlb.transform(split_data)
            col_names = [f"{col}_{cls}" for cls in mlb.classes_]
            output_parts.append(pd.DataFrame(transformed, columns=col_names, index=X.index))
        return pd.concat(output_parts, axis=1)

In [17]:
# --- Custom Transformer: Top K Categories ---
class TopKCategoriesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, top_k=200):
        self.top_k = top_k
        self.top_categories_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            top = X[col].value_counts().nlargest(self.top_k).index
            self.top_categories_[col] = set(top)
        return self

    def transform(self, X):
        X = X.copy()
        for col in X.columns:
            X[col] = X[col].where(X[col].isin(self.top_categories_[col]), other='__other__')
        return X

In [18]:
#final
amenities_weightages = {
    "sea facing": 10,
    "private pool": 10,
    "private jaccuzi": 10,
    "sky villa": 10,
    "helipad": 10,
    "wrap around balcony": 7,
    "infinity swimming pool": 10,
    "high ceiling": 9,
    "located in the heart of city": 10,
    "large open space": 10,
    "skyline view": 10,
    "private terrace/garden": 10,
    "private garage": 10,
    "mansion": 10,
    "club house": 9,
    "large clubhouse": 9,
    "modular kitchen": 9,
    "central ac": 9,
    "banquet hall": 6,
    "premium branded fittings": 9,
    "private garden": 9,
    "full glass wall": 9,
    "garden view": 9,
    "theme based architectures": 9,
    "grand entrance lobby": 9,
    "smart home": 9,
    "library and business centre": 9,
    "recreational pool": 9,
    "projector": 8,
    "swimming pool": 8,
    "gymnasium": 8,
    "indoor squash & badminton courts": 8,
    "outdoor tennis courts": 8,
    "cycling & jogging track": 8,
    "kids play pool with water slides": 8,
    "guest lobby in each floor": 8,
    "aesthetically designed landscape garden": 8,
    "health club with steam / jacuzzi": 8,
    "meditation area": 8,
    "pet park": 8,
    "visitor parking": 8,
    "badminton court": 8,
    "kids play area": 7,
    "community hall": 7,
    "power back up": 7,
    "cctv camera": 7,
    "rain water harvesting": 7,
    "internet/wi-fi connectivity": 7,
    "cycling track": 7,
    "art center": 7,
    "library": 7,
    "fire sprinklers": 7,
    "multipurpose hall": 7,
    "event space & amphitheatre": 7,
    "flower gardens": 6,
    "curated garden": 6,
    "multipurpose courts": 7,
    "dth television facility": 5,
    "fire fighting equipment": 6,
    "provision for power backup": 7,
    "sand pit": 6,
    "sewage treatment plant": 6,
    "solar energy": 7,
    "piped gas": 6,
    "kids club": 6,
    "waste disposal": 6,
    "lift": 5,
    "security": 5,
    "maintenance staff": 5,
    "reserved parking": 5,
    "ro water system": 5,
    "wheelchair accessibility": 5,
    "shopping center": 5,
    "laundry service": 5,
    "bank & atm": 5,
    "community entrance gate": 5,
    "canopy walk": 4,
    "entry exit gate": 4,
    "early learning centre": 4,
    "earth quake resistant": 7,
    "waste water recycling": 6,
    "whiteboard": 3,
    "printer": 3,
    "tea/coffee": 3,
    "house help accommodation": 7,
    "study room": 5,
    "ground water recharging": 5,
    "unknown": 0,
    "3 tier security system": 8,
    "ac in each room": 9,
    "activity deck4": 7,
    "aerobics room": 7,
    "air conditioned": 9,
    "all wooden flooring": 8,
    "arts & craft studio": 6,
    "bar/lounge": 7,
    "barbeque pit": 6,
    "barbeque space": 6,
    "cafeteria/food court": 7,
    "coffee lounge & restaurants": 7,
    "concierge services": 9,
    "conference room": 8,
    "cricket net practice": 6,
    "dance studio": 7,
    "downtown": 10,
    "fingerprint access": 8,
    "fireplace": 6,
    "golf course": 10,
    "hilltop": 10,
    "horticulture": 6,
    "indoor games room": 7,
    "island kitchen layout": 8,
    "jogging and strolling track": 7,
    "kids splash pool": 7,
    "lawn with pathway": 6,
    "guest accommodation":8,
    "marble flooring": 9,
    "mini cinema theatre": 9,
    "half basketball court":7,
    "park": 8,
    "pool with temperature control": 10,
    "intercom facility":6,
    "rentable community space": 6,
    "retail boulevard (retail shops)": 8,
    "service/goods lift": 6,
    "skydeck": 9,
    "vaastu compliant": 7,
    "volleyball court": 6,
    "water front": 10,
    "water storage": 5,
    "water treatment plant": 7,
    "wine cellar": 8
}

In [19]:
class AmenitiesScoreTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='amenities', weightages=None, output_column='assigned_amenities_score'):
        self.column = column
        self.weightages = weightages if weightages is not None else {}
        self.output_column = output_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        def calculate_score(amenities_str):
            # if not isinstance(amenities_str, str):
            #     return 0
            # amenities_list = [a.strip().lower() for a in amenities_str.split(",")]
            # return round(sum(self.weightages.get(a, 0) for a in amenities_list), 2)
            amenities_types = [f.strip().lower() for f in amenities_str.split(",")]
            total_weight = sum(self.weightages.get(f, 0) for f in amenities_types)
            return round(total_weight, 2)

        # Calculate scores
        X[self.output_column] = X[self.column].apply(calculate_score)

        # Replace 0 with NaN
        X[self.output_column] = X[self.output_column].replace(0, pd.NA)
        X[self.output_column] = pd.to_numeric(X[self.output_column], errors='coerce')

        # Drop original amenities column
        X.drop(columns=[self.column], inplace=True)

        return X


In [20]:
# Add missing indicator
class MissingIndicatorAdder(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score'):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column + '_missing'] = X[self.column].isna().astype(int)
        return X

In [21]:
# KNN imputation + MinMax scaling
class ImputeAndScaleAmenity(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score', n_neighbors=5):
        self.column = column
        self.n_neighbors = n_neighbors
        self.imputer = KNNImputer(n_neighbors=n_neighbors)
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        # Fit on the original column
        self.imputer.fit(X[[self.column]])
        imputed = self.imputer.transform(X[[self.column]])
        self.scaler.fit(imputed)
        return self

    def transform(self, X):
        X = X.copy()
        # Impute and scale the column
        imputed = self.imputer.transform(X[[self.column]])
        scaled = self.scaler.transform(imputed)
        # Replace with scaled
        X[self.column] = scaled
        return X

In [22]:
# Custom Ordinal Encoder Wrapper (for single column)
class ConstructionOrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column='construction', categories=None):
        self.column = column
        self.categories = categories
        self.encoder = OrdinalEncoder(categories=self.categories, handle_unknown='use_encoded_value', unknown_value=-1)

    def fit(self, X, y=None):
        self.encoder.fit(X[[self.column]])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = self.encoder.transform(X[[self.column]])
        return X

'builder' - constant / top 200 / target encode   
'project_name' - constant / top 200 / target encode    
'location' - constant / top 200 / target encode    


'project_in_acres' - KNN Imputer    
'area' - KNN Imputer      
'education_mean_km' - KNN Imputer  
'education_min_km' - KNN Imputer  
'transport_mean_km' - KNN Imputer  
'transport_min_km' - KNN Imputer  
'shopping_centre_mean_km' - KNN Imputer  
'shopping_centre_min_km' - KNN Imputer  
'overall_min_mean_km' - KNN Imputer  
'overall_avg_mean_km' - KNN Imputer  
'overall_min_min_km' - KNN Imputer  
'overall_avg_min_km' - KNN Imputer  
'available_units' - KNN Imputer  
'towers' - KNN Imputer  
'flat_on_floor' - KNN Imputer  
'total_floor' - KNN Imputer  
'bath' - KNN Imputer  
'parking' - KNN Imputer  
'commercial_hub_mean_km' - KNN Imputer  
'commercial_hub_min_km' - KNN Imputer  
'balcony' - KNN Imputer  

'lattitude' - iterative imputer   
'longitude' - iterative imputer   

'lift' - median  

'property_type' - mode / OHE  
'status' - mode / ordinal encoding  
'furnish' - mode / ordinal encoding  


'ownership' - constant / OHE  
'facing' - constant / OHE  
'overlooking' - constant / multilable  
'extra_rooms' - constant / multilable  
'flooring' - constant / multilable  


'assigned_amenities_score' -  missingindicator then KNNimputation and then min_max_scale
'construction' -  missingindicator and ordinal_encode

'city' - OHE  
'seller' - OHE  


'education_within_2km' - MinMax Scaling    
'transport_within_2km' - MinMax Scaling     
'shopping_centre_within_2km' - MinMax Scaling    
'commercial_hub_within_2km' - MinMax Scaling  
'hospital_within_2km' - MinMax Scaling  
'tourist_within_2km' - MinMax Scaling  
'total_within_2km' - MinMax Scaling  

In [23]:
impute_topk_target_encoding_cols = ['builder', 'project_name', 'location']

features_to_fill_knn = [
    'project_in_acres','area', 'education_mean_km', 'education_min_km',
    'transport_mean_km','transport_min_km','shopping_centre_mean_km','shopping_centre_min_km',
    'overall_min_mean_km', 'overall_avg_mean_km','overall_min_min_km', 'overall_avg_min_km',
    'available_units','towers','flat_on_floor','total_floor','bath','parking',
    'commercial_hub_mean_km','commercial_hub_min_km','balcony'
] #'costpersqft','emi'

features_to_fill_iterative = ['lattitude','longitude']
features_to_fill_median = ['lift']

impute_mf_and_OHE = ['property_type']

impute_mf_and_ordinal_encode = ['status','furnish']
ordinal_categories = [
    ['under construction', 'ongoing', 'ready to move'],  # status
    ['unfurnished', 'semi-furnished', 'furnished']       # furnish
]

impute_missing_and_OHE = ['ownership', 'facing']

impute_missing_and_multilable = ['overlooking','extra_rooms','flooring']

assignweight_missingindicator_KNNimputation_minmaxscale = ['amenities']

missingindicator_ordinal_encode = ['construction']
construction_categories = [[
    'missing', 'under construction', 'new construction', 'less than 5 years',
    '5 to 10 years', '10 to 15 years', '15 to 20 years', 'above 20 years'
]]

onehotencode = ['city','seller']

min_max_scaling = ['education_within_2km','transport_within_2km','shopping_centre_within_2km',
                   'commercial_hub_within_2km','hospital_within_2km','tourist_within_2km','total_within_2km']

In [24]:
print(X_train.shape)

(9302, 47)


In [25]:
# --- Final Pipeline for Target Encoding Columns ---
builder_location_project_name_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ('top_k', TopKCategoriesTransformer(top_k=200)),
    ('target_encoder', TargetEncoder())
])

property_type_pipeline = Pipeline(steps=[
    ('impute' , SimpleImputer(strategy="most_frequent")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


status_furnish_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="most_frequent")),
    ('ordinal encode', OrdinalEncoder(categories=ordinal_categories,handle_unknown='use_encoded_value',unknown_value=-1 ))
])


ownership_facing_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

overlooking_extra_rooms_flooring_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('multilable', MultiLabelBinarizerTransformer())
])

assigned_amenities_pipeline = Pipeline(steps=[
    ('calculate_score', AmenitiesScoreTransformer(
        column='amenities',
        weightages=amenities_weightages,
        output_column='assigned_amenities_score'
    )),
    ('add_missing_indicator', MissingIndicatorAdder(column='assigned_amenities_score')),
    ('impute_and_scale', ImputeAndScaleAmenity(column='assigned_amenities_score', n_neighbors=5))
])

construction_pipeline = Pipeline(steps=[
    ('add_missing_indicator', MissingIndicatorAdder(column='construction')),
    ('ordinal_encode', ConstructionOrdinalEncoder(column='construction', categories=construction_categories))
])

city_seller_pipeline = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

within2km_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# --- Unified Preprocessor ---
preprocessor = make_column_transformer(
    # impute_constant=missing, topK and Target Encoding
    (builder_location_project_name_pipeline, impute_topk_target_encoding_cols),

    # Imputation
    (KNNImputer(n_neighbors=5), features_to_fill_knn),
    (IterativeImputer(), features_to_fill_iterative),
    (SimpleImputer(strategy="median"), features_to_fill_median),

    # impute = most_frequent and OHE 
    (property_type_pipeline, impute_mf_and_OHE),

    #impute = most_frequent and ordinal encoding
    (status_furnish_pipeline, impute_mf_and_ordinal_encode),

    #impute_constant=missing and OHE
    (ownership_facing_pipeline, impute_missing_and_OHE),

    #impute_constant=missing and multilable
    (overlooking_extra_rooms_flooring_pipeline, impute_missing_and_multilable),

    #missingindicator then KNNimputation and then min_max_scale
    (assigned_amenities_pipeline, assignweight_missingindicator_KNNimputation_minmaxscale),

    #missingindicator and ordinal_encode
    (construction_pipeline, missingindicator_ordinal_encode),

    # onehot_encoder
    (city_seller_pipeline, onehotencode),

    # MinMax Scaling
    (within2km_pipeline, min_max_scaling),

    # Keep other columns
    remainder='passthrough',
    verbose_feature_names_out=False,
    n_jobs=-1
)

# --- Final Pipeline ---
final_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor)
])

In [26]:
# Here, only delete the rows that are outliers in the below columns.
# Even if imputation is applied below, it does NOT impute permanently —
# it only imputes temporarily to help detect and delete outlier data points.

features_to_fill_knn_OD  = [
    'project_in_acres','area', 'education_mean_km', 'education_min_km',                           
    'transport_mean_km','transport_min_km','shopping_centre_mean_km','shopping_centre_min_km',
    'overall_min_mean_km', 'overall_avg_mean_km','overall_min_min_km', 'overall_avg_min_km',
    'available_units','towers','flat_on_floor','total_floor',
    'commercial_hub_mean_km','commercial_hub_min_km','balcony'
]  #OD  means outlier detetion

features_to_fill_iterative_OD = ['lattitude','longitude']

# Impute OD columns separately for LOF
knn_imputer = KNNImputer(n_neighbors=5)
X_train_knn_imp = X_train[features_to_fill_knn_OD].copy()
X_train_knn_imp.loc[:, :] = knn_imputer.fit_transform(X_train_knn_imp)

iter_imputer = IterativeImputer()
X_train_iter_imp = X_train[features_to_fill_iterative_OD].copy()
X_train_iter_imp.loc[:, :] = iter_imputer.fit_transform(X_train_iter_imp)

# Apply LOF separately
lof_knn = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
lof_iter = LocalOutlierFactor(n_neighbors=20, contamination=0.01)

mask_knn = (lof_knn.fit_predict(X_train_knn_imp) == 1)
mask_iter = (lof_iter.fit_predict(X_train_iter_imp) == 1)

final_mask = mask_knn & mask_iter

# Filter train data and target
X_train_filtered = X_train.loc[final_mask].copy()
y_train_filtered = y_train_trans.loc[final_mask].copy()

print(f"Original train size: {X_train.shape[0]}")
print(f"Filtered train size after LOF: {X_train_filtered.shape[0]}")

Original train size: 9302
Filtered train size after LOF: 9114


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\neighbors\_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


In [27]:
X_train_trans = final_pipeline.fit_transform(X_train_filtered, y_train_filtered)
X_test_trans  = final_pipeline.transform(X_test)

#print(X_train_trans.head())
print(X_train_trans.isna().sum()[X_train_trans.isna().sum() > 0])

Series([], dtype: int64)


In [28]:
print(X_train_trans.shape)
print(y_train_filtered.shape)

(9114, 83)
(9114,)


In [29]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

## feature selection using RFECV

In [30]:
# Step Summary:
# 1. Feature selection using RFECV with RandomForestRegressor as the estimator.
# 2. Kept only the selected features from the training and test sets.
# 3. Trained a new RandomForestRegressor on the selected features.
# 4. Evaluated model performance using cross-validation and test metrics.

In [31]:
from sklearn.feature_selection import RFECV

In [32]:
# feature selection using rfecv

rfecv = RFECV(
    estimator=rf,
    step=10,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

In [33]:
# select features

rfecv.fit(X_train_trans, y_train_filtered)

Fitting estimator with 83 features.
Fitting estimator with 73 features.


RFECV(cv=5, estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
      n_jobs=-1, scoring='r2', step=10, verbose=2)

In [34]:
# Check which features were selected
rfecv.get_feature_names_out()

array(['builder', 'project_name', 'location', 'project_in_acres', 'area',
       'education_mean_km', 'education_min_km', 'transport_mean_km',
       'transport_min_km', 'shopping_centre_mean_km',
       'shopping_centre_min_km', 'overall_min_mean_km',
       'overall_avg_mean_km', 'overall_min_min_km', 'overall_avg_min_km',
       'available_units', 'towers', 'flat_on_floor', 'total_floor',
       'bath', 'parking', 'commercial_hub_mean_km',
       'commercial_hub_min_km', 'balcony', 'lattitude', 'longitude',
       'lift', 'property_type_new property', 'property_type_resale',
       'status', 'furnish', 'ownership_co-operative society',
       'ownership_freehold', 'ownership_missing', 'facing_east',
       'facing_missing', 'facing_north - east', 'overlooking_garden/park',
       'overlooking_main road', 'overlooking_missing', 'overlooking_pool',
       'extra_rooms_missing', 'extra_rooms_none of these',
       'extra_rooms_puja', 'extra_rooms_servant', 'extra_rooms_store',
       '

In [35]:
# 1. Transform X data using selected features
X_train_selected = rfecv.transform(X_train_trans)
X_test_selected = rfecv.transform(X_test_trans)


# 2. Train final model
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train_selected, y_train_filtered)


# 3. Predict (in transformed space)
y_pred_train_trans = final_model.predict(X_train_selected)
y_pred_test_trans = final_model.predict(X_test_selected)


# 4. Clip predictions before inverse transform
min_val, max_val = y_train_filtered.min(), y_train_filtered.max()
y_pred_train = pt.inverse_transform(np.clip(y_pred_train_trans, min_val, max_val).reshape(-1, 1)).ravel()
y_pred_test = pt.inverse_transform(np.clip(y_pred_test_trans, min_val, max_val).reshape(-1, 1)).ravel()

y_train_filtered_inv = pt.inverse_transform(y_train_filtered.values.reshape(-1, 1)).ravel()


# 5. Define metric calculation function
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2


train_metrics = calc_metrics(y_train_filtered_inv, y_pred_train)
test_metrics = calc_metrics(y_test, y_pred_test)


# 6. Cross-validation evaluation
scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(lambda y, y_pred: np.sqrt(mean_squared_error(y, y_pred))),
    'R2': make_scorer(r2_score)
}

cv_results = cross_validate(
    final_model,
    X_train_selected,
    y_train_filtered,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    n_jobs=-1
)


# 7. Print results neatly
print("==== Train Metrics (Selected Features) ====")
print(f"MAE: {train_metrics[0]:.4f} | MSE: {train_metrics[1]:.4f} | RMSE: {train_metrics[2]:.4f} | R²: {train_metrics[3]:.4f}")

print("\n==== Test Metrics (Selected Features) ====")
print(f"MAE: {test_metrics[0]:.4f} | MSE: {test_metrics[1]:.4f} | RMSE: {test_metrics[2]:.4f} | R²: {test_metrics[3]:.4f}")

print("\n==== Cross-Validation Train Scores ====")
for key, value in cv_results.items():
    if key.startswith("train"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")

print("\n==== Cross-Validation Test Scores ====")
for key, value in cv_results.items():
    if key.startswith("test"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")

==== Train Metrics (Selected Features) ====
MAE: 0.4752 | MSE: 2.7946 | RMSE: 1.6717 | R²: 0.8315

==== Test Metrics (Selected Features) ====
MAE: 0.5729 | MSE: 3.4255 | RMSE: 1.8508 | R²: 0.7659

==== Cross-Validation Train Scores ====
train_MAE - Mean: 0.1441, Std: 0.0008
train_MSE - Mean: 0.0396, Std: 0.0005
train_RMSE - Mean: 0.1989, Std: 0.0013
train_R2 - Mean: 0.9599, Std: 0.0008

==== Cross-Validation Test Scores ====
test_MAE - Mean: 0.1970, Std: 0.0039
test_MSE - Mean: 0.0741, Std: 0.0026
test_RMSE - Mean: 0.2722, Std: 0.0048
test_R2 - Mean: 0.9247, Std: 0.0044


In [36]:
# # Plot RFECV Scores
# plt.figure(figsize=(10, 5))
# plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
#          rfecv.cv_results_['mean_test_score'])
# plt.xlabel("Number of Selected Features")
# plt.ylabel("Cross-Validation R² Score")
# plt.title("RFECV Feature Selection")
# plt.grid(True)
# plt.tight_layout()
# plt.show()


In [37]:
final_model.feature_importances_

array([2.57372095e-02, 2.70795758e-02, 9.81905061e-02, 2.89903750e-03,
       1.69924379e-01, 4.75810409e-03, 4.01481783e-03, 3.66739369e-03,
       3.69159375e-03, 4.77558978e-03, 5.43541394e-03, 5.01532111e-03,
       3.98200620e-03, 4.56657657e-03, 5.11254792e-03, 2.95122144e-03,
       3.11990778e-03, 8.00911451e-03, 2.69623645e-02, 1.23023994e-01,
       4.00290349e-02, 1.28353656e-02, 9.60333042e-03, 2.64548712e-03,
       4.19982276e-02, 4.16197956e-02, 1.35606435e-03, 5.44318971e-04,
       4.18456715e-04, 6.52283415e-04, 1.50035555e-03, 2.44631754e-04,
       5.86986853e-04, 6.48354342e-04, 3.86375543e-04, 6.34933597e-04,
       1.60574354e-04, 4.09256586e-04, 4.38330343e-04, 7.16037749e-04,
       3.44709564e-04, 3.84613322e-03, 2.40988869e-04, 2.22200379e-04,
       1.13794311e-02, 6.76627852e-04, 4.42959749e-04, 1.73323857e-04,
       1.60490217e-03, 1.36427167e-03, 6.11557690e-04, 1.94565424e-03,
       3.40670291e-03, 5.06930220e-04, 8.23506615e-02, 1.69658749e-02,
      

In [38]:
# # feature importance plot

# (
#     pd.DataFrame(final_model.feature_importances_,
#              index=rfecv.transform(X_train_trans).columns,
#              columns=["importance"])
#     .sort_values(by="importance")
#     .plot(kind='barh',figsize=(10,10))
# )

In [39]:
# Unpack metrics
train_mae, train_mse, train_rmse, train_r2 = train_metrics
test_mae, test_mse, test_rmse, test_r2 = test_metrics

# Log experiment
with mlflow.start_run(run_name="Baseline model with LOF and RFECV"):
    # Log experiment type
    mlflow.log_param("experiment_type", "LOF")

    # Log model parameters
    mlflow.log_params(final_model.get_params())

    # Log train/test evaluation metrics
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_r2", train_r2)

    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    # Log mean cross-validation metrics
    mlflow.log_metric("cv_train_mae", np.mean(cv_results['train_MAE']))
    mlflow.log_metric("cv_train_mse", np.mean(cv_results['train_MSE']))
    mlflow.log_metric("cv_train_rmse", np.mean(cv_results['train_RMSE']))
    mlflow.log_metric("cv_train_r2", np.mean(cv_results['train_R2']))

    mlflow.log_metric("cv_val_mae", np.mean(cv_results['test_MAE']))
    mlflow.log_metric("cv_val_mse", np.mean(cv_results['test_MSE']))
    mlflow.log_metric("cv_val_rmse", np.mean(cv_results['test_RMSE']))
    mlflow.log_metric("cv_val_r2", np.mean(cv_results['test_R2']))

2025/08/12 18:31:29 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



🏃 View run Baseline model with RFECV at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/5/runs/267380293a13401498f427a961955cf0
🧪 View experiment at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/5
